# 12 · Production Concerns: Errors, Retries, Budgets and Testing

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Mostly offline**: failures are
> simulated so they happen on cue. §3 and §5 use Ollama.

In production, model APIs drop connections, agents loop, outputs fail
validation, costs creep up, and a prompt edit silently breaks a route. This
notebook covers the SDK's tools for each, and how to **test** agent code without
calling a model at all.

**You will learn**
- The SDK's **exception types**, and which ones your code should expect
- **Error handlers**: turning `MaxTurnsExceeded` into a graceful answer
- **Model retries**: off by default, and how to turn them on correctly
- Timeouts, and a **token budget** enforced with hooks
- **Testing with a scripted fake model**: deterministic, instant, free
- A production checklist

**Prerequisites:** notebooks 02–10.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import asyncio, httpx, openai
from agents import (Agent, Runner, function_tool, ModelSettings, ModelRetrySettings, RunHooks,
                    Model, ModelResponse, Usage, AgentsException, MaxTurnsExceeded, ModelBehaviorError,
                    UserError, InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered)
from agents.retry import retry_policies
from common import get_model

model = get_model()

## 1 · Why: what goes wrong

| Failure | Example | SDK tool |
|---|---|---|
| Transient API errors | dropped connection, 5xx, rate limit | `ModelSettings(retry=ModelRetrySettings(...))` |
| Runaway loops | an agent re-checking a status forever | `max_turns` + an `error_handlers` fallback |
| Invalid output | JSON that doesn't match `output_type` | `ModelBehaviorError`; `error_handlers` |
| Hanging calls | a slow tool or model | tool `timeout=`, model `timeout` setting |
| Cost creep | long loops, big contexts | usage tracking + a budget (hooks) |
| Silent regressions | a prompt edit breaks routing | tests with a fake model |

## 2 · The exception types

Everything the SDK raises derives from `AgentsException`. The ones you'll handle:

| Exception | Raised when | Typical handling |
|---|---|---|
| `MaxTurnsExceeded` | the loop hit `max_turns` | an error handler, or a "couldn't finish" reply |
| `ModelBehaviorError` | the model produced something invalid (bad JSON, unknown tool) | retry, or fall back |
| `InputGuardrailTripwireTriggered` / `OutputGuardrailTripwireTriggered` | a guardrail tripped | a deliberate refusal (notebook 07) |
| `UserError` | *your* setup is wrong (bad schema, a tool raising with `failure_error_function=None`) | fix the code |
| provider errors (e.g. `openai.APIConnectionError`) | the model API failed after retries | retry later, alert |

A robust entry point catches them deliberately rather than letting a stack trace
reach the user:

```python
try:
    result = await Runner.run(agent, message, max_turns=8)
    reply = result.final_output
except InputGuardrailTripwireTriggered:
    reply = "Sorry, I can't help with that."
except MaxTurnsExceeded:
    reply = "That took too many steps; a human will follow up."
except openai.APIConnectionError:
    reply = "Our assistant is temporarily unavailable."
```

## 3 · Error handlers: a graceful answer instead of an exception

For some failures you'd rather **finish the run** with a sensible final output.
`Runner.run(..., error_handlers={...})` maps an error kind (`max_turns`,
`invalid_final_output`, `model_refusal`) to a function. Whatever the function
returns becomes the final output. It receives the error and the run's data (the
items so far, the last agent), so it can report partial progress:

In [ ]:
@function_tool
def next_step(n: int) -> int:
    """Return the step after n."""
    return n + 1

counter = Agent(name='Counter', model=model, tools=[next_step],
                instructions='Start at 0. Call next_step with the current number, one call at a time, until 10.')


def turn_limit_reached(data):
    steps = sum(type(i).__name__ == 'ToolCallOutputItem' for i in data.run_data.new_items)
    return f'Stopped at the turn limit after {steps} steps. Partial result kept; flagged for review.'


result = await Runner.run(counter, 'Go.', max_turns=3, error_handlers={'max_turns': turn_limit_reached})
print('final_output:', result.final_output)

No exception: the run ended with a clear, honest final output built by your
code, just as in LangGraph's error handlers.

## 4 · Model retries: off until you turn them on

A dropped connection to the model API shouldn't fail a user's request. The SDK
can retry model calls, but **by default it doesn't**. We'll prove that with a
deliberately flaky model: a wrapper that raises a connection error twice before
delegating to the real model.

In [ ]:
class FlakyModel(Model):
    # Fails with a connection error `fail_times` times, then behaves like `inner`.

    def __init__(self, inner: Model, fail_times: int):
        self.inner, self.remaining, self.calls = inner, fail_times, 0

    async def get_response(self, *args, **kwargs):
        self.calls += 1
        if self.remaining > 0:
            self.remaining -= 1
            raise openai.APIConnectionError(request=httpx.Request('POST', 'http://model.invalid'))
        return await self.inner.get_response(*args, **kwargs)

    def stream_response(self, *args, **kwargs):
        return self.inner.stream_response(*args, **kwargs)


async def try_with(settings: ModelSettings, label: str):
    flaky = FlakyModel(model, fail_times=2)
    agent = Agent(name='A', model=flaky, model_settings=settings, instructions='Reply with OK.')
    try:
        result = await Runner.run(agent, 'Say OK')
        print(f'{label:<34} succeeded after {flaky.calls} model calls: {result.final_output!r}')
    except openai.APIConnectionError:
        print(f'{label:<34} FAILED after {flaky.calls} model call(s)')


await try_with(ModelSettings(), 'default settings')
await try_with(ModelSettings(retry=ModelRetrySettings(max_retries=3, policy=retry_policies.provider_suggested())),
               'retries, provider_suggested only')
await try_with(ModelSettings(retry=ModelRetrySettings(
                   max_retries=3,
                   policy=retry_policies.any(retry_policies.network_error(), retry_policies.provider_suggested())
               )), 'retries, network errors + provider')

Three results, three lessons:

1. **Default: no retries.** One connection blip fails the request.
2. **A retry count isn't enough; the policy decides what's retryable.**
   `provider_suggested()` follows hints from the provider (like a `Retry-After`
   header). A plain connection error carries none, so it wasn't retried.
3. **`network_error()` covers dropped connections.** Combine policies with
   `retry_policies.any(...)`. There are also `http_status(...)` (e.g. 429, 5xx)
   and `retry_after()`.

Set retries once, on `RunConfig(model_settings=...)` or each agent. Retries
repeat *model calls*, not tool calls, so tools still need their own error
handling (notebook 04).

## 5 · Timeouts and a token budget

**Timeouts.** Tools take `timeout=` (notebook 04). The model call has
`ModelSettings(timeout=seconds)`, a network timeout for each request, which pairs
naturally with retries.

**Budgets.** `max_turns` limits model *calls*, not *cost*. A budget in tokens
is easy to enforce with a hook (notebook 10) that counts usage after every model
call and raises when the limit is exceeded:

In [ ]:
class BudgetExceeded(Exception):
    pass


class TokenBudget(RunHooks):
    def __init__(self, limit: int):
        self.limit, self.used = limit, 0

    async def on_llm_end(self, context, agent, response):
        self.used += response.usage.total_tokens
        if self.used > self.limit:
            raise BudgetExceeded(f'{self.used} tokens used, budget {self.limit}')


writer = Agent(name='Writer', model=model, instructions='Write detailed, long answers.')
budget = TokenBudget(limit=150)
try:
    await Runner.run(writer, 'Explain in depth how delivery robots navigate cities.', hooks=budget)
except BudgetExceeded as err:
    print('stopped:', err)

The budget is checked after each model call, so a single call can overshoot it:
a budget stops the *next* call, it doesn't truncate the current one (use
`max_tokens` for that). Multiply tokens by your provider's prices for a cost
budget.

## 6 · Testing with a fake model

LLM output varies, so don't test *the model*; test **your code around it**. Does
the router pick the right specialist for each classification? Does the agent
call the tool with the right arguments? Does the fallback work when the model
rambles?

A **fake model** implements the SDK's `Model` interface and returns **scripted**
responses, either text or tool calls, in order. Tests become deterministic,
instant and free:

In [ ]:
from openai.types.responses import ResponseOutputMessage, ResponseOutputText, ResponseFunctionToolCall


class ScriptedModel(Model):
    # Returns scripted outputs in order: a str is a message; a (tool_name, json_args) tuple is a tool call.

    def __init__(self, outputs):
        self.outputs, self.calls = list(outputs), 0

    async def get_response(self, *args, **kwargs):
        out = self.outputs[self.calls]
        self.calls += 1
        if isinstance(out, str):
            item = ResponseOutputMessage(id='msg', type='message', role='assistant', status='completed',
                                         content=[ResponseOutputText(type='output_text', text=out, annotations=[])])
        else:
            name, arguments = out
            item = ResponseFunctionToolCall(id='fc', type='function_call', call_id=f'call-{self.calls}',
                                            name=name, arguments=arguments)
        return ModelResponse(output=[item], usage=Usage(), response_id=None)

    def stream_response(self, *args, **kwargs):
        raise NotImplementedError

Now tests for the kind of code you wrote in notebooks 04–06. The key design
choice, as in LangGraph's section: **build agents in functions that take the
model as a parameter**, so tests can pass a fake one.

In [ ]:
refunds = []

@function_tool
def issue_refund(order_id: str) -> str:
    """Refund an order."""
    refunds.append(order_id)
    return f'refunded {order_id}'


def build_billing(model):                                  # production passes get_model()
    return Agent(name='Billing', model=model, tools=[issue_refund],
                 instructions='Refund orders with issue_refund.')


async def test_refund_calls_the_tool_with_the_order_id():
    refunds.clear()
    fake = ScriptedModel([('issue_refund', '{"order_id": "A-17"}'), 'Your refund is on its way.'])
    result = await Runner.run(build_billing(fake), 'Refund A-17 please.')
    assert refunds == ['A-17'], refunds
    assert result.final_output == 'Your refund is on its way.'


async def test_claim_without_action_is_detected():
    # The model SAYS it refunded but never calls the tool: our verification (notebook 06) must catch it.
    refunds.clear()
    fake = ScriptedModel(['Done! Your refund has been processed.'])
    result = await Runner.run(build_billing(fake), 'Refund A-17 please.')
    called = any(type(i).__name__ == 'ToolCallItem' for i in result.new_items)
    assert not called and refunds == [], 'the fake never called the tool'


async def test_router_falls_back_on_rambling():
    # Code-orchestrated routing (notebook 05): a rambling classifier must route to "general".
    fake = ScriptedModel(['Hmm, hard to say, maybe billing?'])
    reply = (await Runner.run(Agent(name='Router', model=fake), 'x')).final_output.strip().lower()
    route = reply if reply in ('billing', 'technical') else 'general'
    assert route == 'general'


for test in (test_refund_calls_the_tool_with_the_order_id,
             test_claim_without_action_is_detected,
             test_router_falls_back_on_rambling):
    await test()
    print('✅', test.__name__)

These tests run in milliseconds with no model and no network, so they can run
on every commit. Three layers of testing fit together:

| Layer | Checks | Uses |
|---|---|---|
| **Unit / route tests** | your code: routing, tool wiring, fallbacks, verification | a scripted fake model |
| **Tool tests** | each function tool on its own | plain function calls |
| **Evaluations** | the *quality* of real model output on a fixed set of cases | the real model, scored |

## 7 · Running it for real

- **One entry point per request** that catches the exceptions above and always
  returns a deliberate reply.
- **Sessions in a real database** (SQLAlchemy/Redis backends, notebook 09), keyed by
  conversation; store `last_agent` alongside.
- **Paused runs** (`RunState.to_json()`) in durable storage, with an expiry policy
  for approvals nobody answers.
- **Concurrency.** Agents are stateless, so one process can serve many runs
  concurrently with `asyncio`. Put per-user data in context, never on agents.
- **Observability.** A trace processor to your tracing backend (notebook 10), with
  a decision about sensitive data.
- **Debugging locally.** `agents.enable_verbose_stdout_logging()` prints what the SDK
  is doing, request by request.

## 8 · Production checklist

- [ ] Every run has `max_turns` and an error handler or a caught `MaxTurnsExceeded`
- [ ] Model retries configured, with a policy that includes network errors
- [ ] Timeouts on tools (async) and model requests
- [ ] Guardrails **blocking** where tools have side effects (notebook 07)
- [ ] Approvals for irreversible tools, with paused runs stored durably (notebook 08)
- [ ] Side effects **verified** from run items or the system of record (notebook 06)
- [ ] Idempotent side-effecting tools (crash re-runs, notebook 09)
- [ ] A token/cost budget, and usage logged per run
- [ ] Least-privilege tools (`is_enabled`, MCP tool filters)
- [ ] Tracing with a deliberate data policy
- [ ] Route and tool tests with a fake model, run in CI
- [ ] Local-model quirks handled (leaked tool calls, notebook 11)

## 9 · Check yourself

<details><summary><b>1.</b> What do you get by default when a model call hits a dropped connection?</summary>

An exception (e.g. `openai.APIConnectionError`): the SDK doesn't retry model calls unless you configure `ModelSettings(retry=ModelRetrySettings(...))`.
</details>

<details><summary><b>2.</b> You set <code>max_retries=3</code> but connection errors still fail immediately. Why?</summary>

The retry policy decides what's retryable. `provider_suggested()` only follows provider hints, which a raw connection error doesn't have. Include `retry_policies.network_error()` (e.g. via `retry_policies.any(...)`).
</details>

<details><summary><b>3.</b> How do you turn <code>MaxTurnsExceeded</code> into a normal final output?</summary>

Pass `error_handlers={'max_turns': fn}` to `Runner.run`. The function receives the error and the run's data, and its return value becomes the final output.
</details>

<details><summary><b>4.</b> Why build agents inside functions that take a model?</summary>

So tests can pass a scripted fake model and check your code's behaviour (tool arguments, routing, fallbacks, verification) deterministically and without cost, while production passes the real model.
</details>

<details><summary><b>5.</b> Why can a token budget enforced in <code>on_llm_end</code> overshoot?</summary>

It's checked after each model call completes, so the call that crosses the limit has already happened. It prevents further calls; `max_tokens` caps the length of each individual response.
</details>

## Takeaway

Expect failure and decide how each one ends: catch the SDK's exceptions at the
entry point, turn loop limits into graceful answers with **error handlers**,
**enable model retries** with a policy that covers network errors, add timeouts
and a token budget, and verify side effects. Build agents from factories so a
**scripted fake model** can test your logic on every commit.

That completes the teaching notebooks. Next → `project.ipynb`: the Nimbus Briefing
Assistant, built with the Agents SDK.